# Whole Census Geocoder + Tract Matcher Analysis

This notebook analyzes match performance across the whole-dataset, reading the **post-tract-matcher** output (`5_whole_census_tract_matcher_*.csv`). It reports both the original geocoder outcomes (`match`: Match / No_Match / Tie / Missing) and the downstream tract resolution outcomes (`tract_resolution_method`, `tract_resolved`).

The state-level US map shows the tract resolution rate by state — i.e., the share of CoStar properties in each state that ultimately resolved to a single census tract.

In [1]:
import glob
import os
import re

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

DATA_GLOB = "../../data/1_derived/5_whole_census_tract_matcher_0*.csv"
files = sorted(glob.glob(DATA_GLOB))
if not files:
    raise FileNotFoundError(f"No files found for pattern: {DATA_GLOB}")

print(f"Found {len(files)} tract-matcher batch files")
for f in files:
    print(f" - {os.path.basename(f)}")

# Load only the columns this notebook actually uses, to keep memory usage bounded.
NEEDED_COLS = [
    "PropertyId",
    "full_address_std",
    "match",
    "matched_address",
    "latitude",
    "longitude",
    "state_fips",
    "county_fips",
    "geoid",
    "tie_geoids",
    "tie_matched_addresses",
    "match_type",
    "zip_match_detail",
    "tract_resolution_method",
    "final_census_tract",
    "tract_resolved",
]
first_cols = pd.read_csv(files[0], nrows=0).columns.tolist()
use_cols = [c for c in NEEDED_COLS if c in first_cols]

df = pd.concat(
    (pd.read_csv(f, usecols=use_cols, low_memory=False) for f in files),
    ignore_index=True,
)
print(f"\nCombined shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()

Found 7 tract-matcher batch files
 - 5_whole_census_tract_matcher_001.csv
 - 5_whole_census_tract_matcher_002.csv
 - 5_whole_census_tract_matcher_003.csv
 - 5_whole_census_tract_matcher_004.csv
 - 5_whole_census_tract_matcher_005.csv
 - 5_whole_census_tract_matcher_006.csv
 - 5_whole_census_tract_matcher_007.csv



Combined shape: 1,051,219 rows x 16 columns


,PropertyId,full_address_std,match,matched_address,latitude,longitude,state_fips,county_fips,geoid,tie_matched_addresses,tie_geoids,match_type,zip_match_detail,tract_resolution_method,final_census_tract,tract_resolved
0,435558.0,"4501 CIRCLE 75 PKY SE, ATLANTA, GA 30339, UNIT...",No_Match,NaN,NaN,NaN,0.0,0.0,0.000000e+00,NaN,NaN,One-to-One,Geocoder ZIP Missing,One-to-One Primary,0.0,True
1,445018.0,"3200 DOWNWOOD CIR NW, ATLANTA, GA 30327, UNITE...",Match,"3200 DOWNWOOD CIR NW, ATLANTA, GA, 30327",33.840893,-84.426522,13.0,121.0,1.312101e+14,NaN,NaN,One-to-One,One-to-One Matched,One-to-One Primary,9803.0,True
2,440921.0,"100 EDGEWOOD AVE NE, ATLANTA, GA 30303, UNITED...",Match,"100 EDGEWOOD AVE NE, ATLANTA, GA, 30303",33.754541,-84.386374,13.0,121.0,1.312101e+14,NaN,NaN,One-to-One,One-to-One Matched,One-to-One Primary,11901.0,True
3,442552.0,"1888 EMERY ST NW, ATLANTA, GA 30318, UNITED ST...",Match,"1888 EMERY ST NW, ATLANTA, GA, 30318",33.807229,-84.414403,13.0,121.0,1.312101e+14,NaN,NaN,One-to-One,One-to-One Matched,One-to-One Primary,9001.0,True
4,444958.0,"3500 LENOX RD NE, ATLANTA, GA 30326, UNITED ST...",Match,"3500 LENOX RD, ATLANTA, GA, 30326",33.848737,-84.360784,13.0,121.0,1.312101e+14,NaN,NaN,One-to-One,One-to-One Matched,One-to-One Primary,9606.0,True


## Overall Match + Tract Resolution Performance

Two views of "match rate":
- **Geocoder match rate** — share of rows whose Census Geocoder returned a confident single match (`match == 'Match'`).
- **Tract resolution rate** — share of rows that ultimately resolved to a single census tract, including ties that were collapsed by the tract matcher.

In [2]:
df["match"] = df.get("match", pd.Series(index=df.index, dtype="object")).fillna("Missing")

is_match = df["match"].eq("Match")
is_tie = df["match"].eq("Tie")
is_matched_or_tie = is_match | is_tie
is_resolved = df["tract_resolved"].astype(bool) if "tract_resolved" in df.columns else df["final_census_tract"].notna()

overall = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Geocoder Match rows",
            "Geocoder Tie rows",
            "Geocoder Matched-or-Tie rows",
            "Geocoder strict match rate (%)",
            "Geocoder matched-or-tie rate (%)",
            "Tract-resolved rows",
            "Tract resolution rate (%)",
        ],
        "value": [
            len(df),
            int(is_match.sum()),
            int(is_tie.sum()),
            int(is_matched_or_tie.sum()),
            round(is_match.mean() * 100, 2),
            round(is_matched_or_tie.mean() * 100, 2),
            int(is_resolved.sum()),
            round(is_resolved.mean() * 100, 2),
        ],
    }
)

match_dist = (
    df["match"]
    .value_counts(dropna=False)
    .rename_axis("match")
    .reset_index(name="count")
)
match_dist["pct"] = (match_dist["count"] / len(df) * 100).round(2)

tract_dist = (
    df["tract_resolution_method"]
    .value_counts(dropna=False)
    .rename_axis("tract_resolution_method")
    .reset_index(name="count")
)
tract_dist["pct"] = (tract_dist["count"] / len(df) * 100).round(2)

display(overall)
display(match_dist)
display(tract_dist)

,metric,value
0,Total rows,1051219.00
1,Geocoder Match rows,970482.00
2,Geocoder Tie rows,6697.00
3,Geocoder Matched-or-Tie rows,977179.00
4,Geocoder strict match rate (%),92.32
5,Geocoder matched-or-tie rate (%),92.96
6,Tract-resolved rows,1047897.00
7,Tract resolution rate (%),99.68


,match,count,pct
0,Match,970482,92.32
1,No_Match,73355,6.98
2,Tie,6697,0.64
3,Missing,685,0.07


,tract_resolution_method,count,pct
0,One-to-One Primary,1043837,99.30
1,Tie Resolved via ZIP-Matching Ties,3746,0.36
2,Tract Ambiguous,2637,0.25
3,Tract Missing,685,0.07
4,Tie Resolved via All Ties Unanimous,314,0.03


## State-Level Tract Resolution Rates and US Map

For each state, we compute the share of CoStar properties that resolved to a single census tract (`tract_resolved == True`). The choropleth below shows this rate; the table below it ranks states.

In [3]:
fips_to_abbrev = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO", "09": "CT",
    "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI", "16": "ID", "17": "IL",
    "18": "IN", "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME", "24": "MD",
    "25": "MA", "26": "MI", "27": "MN", "28": "MS", "29": "MO", "30": "MT", "31": "NE",
    "32": "NV", "33": "NH", "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA", "54": "WV",
    "55": "WI", "56": "WY", "60": "AS", "66": "GU", "69": "MP", "72": "PR", "78": "VI"
}

if "state_fips" in df.columns:
    state_fips_clean = (
        pd.to_numeric(df["state_fips"], errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(2)
    )
else:
    state_fips_clean = pd.Series([""] * len(df), index=df.index)

state_from_fips = state_fips_clean.map(fips_to_abbrev)

if "full_address_std" in df.columns:
    state_from_address = df["full_address_std"].astype(str).str.extract(
        r",\s*([A-Z]{2})\s+\d{5}(?:-\d{4})?(?:,|$)", expand=False
    )
else:
    state_from_address = pd.Series([pd.NA] * len(df), index=df.index, dtype="string")

df_state = df.copy()
state_abbrev = state_from_fips.combine_first(state_from_address)
state_abbrev = pd.Series(state_abbrev, index=df.index, dtype="string").str.strip().str.upper()
state_abbrev = state_abbrev.where(state_abbrev.str.match(r"^[A-Z]{2}$", na=False), pd.NA)
df_state["state_abbrev"] = state_abbrev

state_summary = (
    df_state.dropna(subset=["state_abbrev"])
    .groupby("state_abbrev", as_index=False)
    .agg(
        total_rows=("state_abbrev", "size"),
        match_rows=("match", lambda s: (s == "Match").sum()),
        tie_rows=("match", lambda s: (s == "Tie").sum()),
        resolved_rows=("tract_resolved", "sum"),
        ambiguous_rows=("tract_resolution_method", lambda s: (s == "Tract Ambiguous").sum()),
        missing_rows=("tract_resolution_method", lambda s: (s == "Tract Missing").sum()),
    )
)

for c in ["total_rows", "match_rows", "tie_rows", "resolved_rows", "ambiguous_rows", "missing_rows"]:
    state_summary[c] = pd.to_numeric(state_summary[c], errors="coerce").fillna(0).astype(int)

state_summary["strict_match_rate_pct"] = (
    state_summary["match_rows"] / state_summary["total_rows"] * 100
).round(2)
state_summary["match_or_tie_rate_pct"] = (
    (state_summary["match_rows"] + state_summary["tie_rows"]) / state_summary["total_rows"] * 100
).round(2)
state_summary["tract_resolution_rate_pct"] = (
    state_summary["resolved_rows"] / state_summary["total_rows"] * 100
).round(2)
state_summary = state_summary.sort_values(
    ["tract_resolution_rate_pct", "total_rows"], ascending=[False, False]
)

# Limit map to 50 states + DC so the color scale is meaningful.
us_states = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA",
    "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM",
    "NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA",
    "WV","WI","WY",
}
map_df = state_summary[state_summary["state_abbrev"].isin(us_states)].copy()

# Color scheme matches the April 2026 PDF: RdYlGn (red = low, yellow = mid, green = high)
# with auto color range so the gradient spans the actual min/max of the data.
fig = px.choropleth(
    map_df,
    locations="state_abbrev",
    locationmode="USA-states",
    color="tract_resolution_rate_pct",
    hover_name="state_abbrev",
    hover_data={
        "total_rows": ":,",
        "match_rows": ":,",
        "tie_rows": ":,",
        "resolved_rows": ":,",
        "ambiguous_rows": ":,",
        "missing_rows": ":,",
        "strict_match_rate_pct": ":.2f",
        "tract_resolution_rate_pct": ":.2f",
    },
    scope="usa",
    color_continuous_scale="RdYlGn",
    title="Census Tract Resolution Rate by State (%)",
)
fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))
fig.show()

state_summary.head(15)

,state_abbrev,total_rows,match_rows,tie_rows,resolved_rows,ambiguous_rows,missing_rows,strict_match_rate_pct,match_or_tie_rate_pct,tract_resolution_rate_pct
21,ME,2975,2957,18,2973,2,0,99.39,100.00,99.93
30,NH,3782,3768,14,3779,3,0,99.63,100.00,99.92
0,AK,1058,1016,3,1057,1,0,96.03,96.31,99.91
3,AZ,24670,23373,60,24646,15,9,94.74,94.99,99.90
16,KS,7596,7296,32,7585,7,4,96.05,96.47,99.86
29,NE,4953,4666,17,4946,3,4,94.21,94.55,99.86
50,WY,722,656,3,721,1,0,90.86,91.27,99.86
33,NV,11878,11194,50,11860,13,5,94.24,94.66,99.85
45,VA,25455,23723,79,25414,31,10,93.20,93.51,99.84
20,MD,18351,17116,77,18320,27,4,93.27,93.69,99.83


## Data Quality by Match Status

Coverage of `latitude`, `longitude`, `geoid`, and `final_census_tract` segmented by geocoder match status.

In [4]:
quality_cols = [c for c in ["latitude", "longitude", "geoid", "final_census_tract"] if c in df.columns]

quality_by_match = (
    df.groupby("match", dropna=False)[quality_cols]
    .apply(lambda x: x.notna().mean() * 100)
    .round(2)
    .reset_index()
)

overall_missing = pd.DataFrame({
    "column": quality_cols,
    "missing_count": [df[c].isna().sum() for c in quality_cols],
    "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in quality_cols],
})

display(quality_by_match)
display(overall_missing)

,match,latitude,longitude,geoid,final_census_tract
0,Match,100.0,100.0,100.0,100.00
1,Missing,0.0,0.0,0.0,0.00
2,No_Match,0.0,0.0,100.0,100.00
3,Tie,100.0,100.0,100.0,60.62


,column,missing_count,missing_pct
0,latitude,74040,7.04
1,longitude,74040,7.04
2,geoid,685,0.07
3,final_census_tract,3322,0.32


## High-Volume County Patterns

In [5]:
county_cols_ok = all(c in df.columns for c in ["state_fips", "county_fips", "match"])
if county_cols_ok:
    county_df = df.copy()
    county_df["state_fips"] = pd.to_numeric(county_df["state_fips"], errors="coerce").astype("Int64").astype(str).str.replace("<NA>", "", regex=False).str.zfill(2)
    county_df["county_fips"] = pd.to_numeric(county_df["county_fips"], errors="coerce").astype("Int64").astype(str).str.replace("<NA>", "", regex=False).str.zfill(3)
    county_df["county_geoid"] = county_df["state_fips"] + county_df["county_fips"]
    county_df = county_df[county_df["county_geoid"].str.len() == 5]

    county_summary = (
        county_df.groupby("county_geoid", as_index=False)
        .agg(
            total_rows=("county_geoid", "size"),
            match_rows=("match", lambda s: (s == "Match").sum()),
            tie_rows=("match", lambda s: (s == "Tie").sum()),
            resolved_rows=("tract_resolved", "sum"),
        )
    )
    county_summary["strict_match_rate_pct"] = (county_summary["match_rows"] / county_summary["total_rows"] * 100).round(2)
    county_summary["tract_resolution_rate_pct"] = (county_summary["resolved_rows"] / county_summary["total_rows"] * 100).round(2)
    county_summary["state_abbrev"] = county_summary["county_geoid"].str[:2].map(fips_to_abbrev)

    top_volume = county_summary.sort_values("total_rows", ascending=False).head(20)
    top_volume
else:
    print("County-level columns were not found in the dataset.")

## Tie Ambiguity Diagnostics

For each tie row, count the distinct candidate GEOIDs the geocoder returned, and cross with the tract resolution outcome to see which ties were collapsed to a single tract vs left ambiguous.

In [6]:
tie_diag_cols = [c for c in ["PropertyId", "full_address_std", "tie_geoids", "tie_matched_addresses", "tract_resolution_method", "final_census_tract"] if c in df.columns]

tie_df = df[df["match"] == "Tie"].copy()
if not tie_df.empty and "tie_geoids" in tie_df.columns:
    tie_df["n_tie_geoids"] = tie_df["tie_geoids"].fillna("").astype(str).apply(
        lambda s: len({x.strip() for x in s.split("|") if x.strip()})
    )

    tie_summary = tie_df["n_tie_geoids"].value_counts().sort_index().rename_axis("candidate_geoids").reset_index(name="row_count")
    tie_summary["pct_of_ties"] = (tie_summary["row_count"] / len(tie_df) * 100).round(2)

    tie_resolution_summary = (
        tie_df["tract_resolution_method"]
        .value_counts(dropna=False)
        .rename_axis("tract_resolution_method")
        .reset_index(name="row_count")
    )
    tie_resolution_summary["pct_of_ties"] = (tie_resolution_summary["row_count"] / len(tie_df) * 100).round(2)

    display_cols = tie_diag_cols + ["n_tie_geoids"]

    display(tie_summary)
    display(tie_resolution_summary)
    display(tie_df[display_cols].sort_values("n_tie_geoids", ascending=False).head(25))
else:
    print("No Tie rows or tie_geoids column unavailable.")

,candidate_geoids,row_count,pct_of_ties
0,1,1945,29.04
1,2,4510,67.34
2,3,191,2.85
3,4,40,0.60
4,5,8,0.12
5,6,2,0.03
6,7,1,0.01


,tract_resolution_method,row_count,pct_of_ties
0,Tie Resolved via ZIP-Matching Ties,3746,55.94
1,Tract Ambiguous,2637,39.38
2,Tie Resolved via All Ties Unanimous,314,4.69


,PropertyId,full_address_std,tie_geoids,tie_matched_addresses,tract_resolution_method,final_census_tract,n_tie_geoids
350641,8907511.0,"261 MARION OAKS BLVD, OCALA, FL 34473, UNITED ...",120830010131083 | 120830010131067 | 1208300101...,"261 MARION OAKS LN, OCALA, FL, 34473 | 261 MAR...",Tract Ambiguous,NaN,7
44840,1039645.0,"1030 COUNTY RD E W, SAINT PAUL, MN 55126, UNIT...",271230421022000 | 271230423024003 | 2712304230...,"1030 CO RD D E, SAINT PAUL, MN, 55109 | 1030 C...",Tie Resolved via ZIP-Matching Ties,40706.0,6
45046,1040714.0,"1000 COUNTY RD E W, SAINT PAUL, MN 55126, UNIT...",271230421022000 | 271230423024003 | 2712304230...,"1000 CO RD D E, SAINT PAUL, MN, 55109 | 1000 C...",Tie Resolved via ZIP-Matching Ties,40706.0,6
85612,10008708.0,"9 N MAIN ST, WILKES BARRE, PA 18711, UNITED ST...",420792001003011 | 420792119004005 | 4207921510...,"9 N MAIN ST, WILKES BARRE, PA, 18701 | 9 N MAI...",Tract Ambiguous,NaN,5
225921,5941890.0,"617 MAIN ST, PEORIA, IL 61602, UNITED STATES",171430048013006 | 171430018002003 | 1717902040...,"617 S MAIN ST, PEORIA, IL, 61604 | 617 W MAIN ...",Tract Ambiguous,NaN,5
297173,8781268.0,"18641 COUNTY RD N, NAPOLEON, OH 43545, UNITED ...",390690002002010 | 390690002002005 | 3906900020...,"18641 CO RD P, NAPOLEON, OH, 43545 | 18641 CO ...",Tie Resolved via ZIP-Matching Ties,200.0,5
873717,10177262.0,"2700 NE LOOP 820, FORT WORTH, TX 76137, UNITED...",484391013012018 | 484391056004011 | 4843910600...,"2700 E LOOP 820, FORT WORTH, TX, 76112 | 2700 ...",Tract Ambiguous,NaN,5
870897,10985234.0,"2800 NE LOOP 820, FORT WORTH, TX 76137, UNITED...",484391013012018 | 484391056004011 | 4843910600...,"2800 E LOOP 820, FORT WORTH, TX, 76112 | 2800 ...",Tract Ambiguous,NaN,5
698750,1214120.0,"1A MAIN ST, EAST HARTFORD, CT 6118, UNITED STATES",090035003001006 | 090034921002026 | 0900349450...,"1A MAIN ST, HARTFORD, CT, 06106 | 1A MAIN ST, ...",Tract Ambiguous,NaN,5
702739,1214101.0,"3 MAIN ST, EAST HARTFORD, CT 6118, UNITED STATES",090035003001006 | 090034921002026 | 0900349450...,"3 MAIN ST, HARTFORD, CT, 06106 | 3 MAIN ST, HA...",Tract Ambiguous,NaN,5


## Export Analysis Outputs

In [7]:
OUT_DIR = "../../output/2_analysis"
TABLES_DIR = os.path.join(OUT_DIR, "tables")
FIGURES_DIR = os.path.join(OUT_DIR, "figures")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

overall.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_overall_summary.csv"), index=False)
match_dist.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_match_distribution.csv"), index=False)
tract_dist.to_csv(os.path.join(TABLES_DIR, "whole_tract_resolution_method_distribution.csv"), index=False)
state_summary.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_state_summary.csv"), index=False)
overall_missing.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_missing_summary.csv"), index=False)

if 'county_summary' in locals():
    county_summary.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_county_summary.csv"), index=False)

if 'tie_summary' in locals():
    tie_summary.to_csv(os.path.join(TABLES_DIR, "whole_geocoder_tie_summary.csv"), index=False)
if 'tie_resolution_summary' in locals():
    tie_resolution_summary.to_csv(os.path.join(TABLES_DIR, "whole_tract_resolution_tie_summary.csv"), index=False)

map_html_path = os.path.join(FIGURES_DIR, "whole_geocoder_state_match_rate_map.html")
map_png_path = os.path.join(FIGURES_DIR, "whole_geocoder_state_match_rate_map.png")
fig.write_html(map_html_path, include_plotlyjs="cdn")
try:
    fig.write_image(map_png_path, width=1100, height=650, scale=2)
    print(f"Saved PNG to {map_png_path}")
except Exception as e:
    print(f"PNG export skipped: {e}")

print(f"Saved tables to: {TABLES_DIR}")
print(f"Saved figures to: {FIGURES_DIR}")

Saved PNG to ../../output/2_analysis\figures\whole_geocoder_state_match_rate_map.png
Saved tables to: ../../output/2_analysis\tables
Saved figures to: ../../output/2_analysis\figures
